# Lab 4：Deep Research Agent

## 學習目標
1. 用 **Planner** 把大問題拆成子問題
2. 用 **Reader** 摘要每個子問題的資料
3. 用 **Synthesizer** 合併成完整報告

## 流程
```
研究題目 → Planner（拆子問題）→ Reader（摘要各子問題）→ Synthesizer（合併報告）
```

In [ ]:
%%capture
!pip install langchain-openai langchain-core

In [ ]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

os.environ["OPENAI_API_KEY"] = "your-api-key-here"

llm = ChatOpenAI(model="gpt-5.4-mini-2026-03-17", temperature=0)

# 模擬搜尋資料庫（實際場景可接 Tavily 或 Google Search API）
MOCK_DATA = {
    "硬體規格": "HP EliteBook 855 搭載 AMD Ryzen 7 PRO，32GB DDR5；ThinkPad X1 Carbon 搭載 Intel Core Ultra，最高 64GB LPDDR5。",
    "安全性": "HP EliteBook 提供 HP Wolf Security、指紋辨識；ThinkPad 提供 ThinkShield 安全套件、Match-on-Chip 指紋辨識。",
    "企業管理": "HP 支援 HP Manageability 與 MIL-STD-810H；Lenovo 提供 ThinkShield Endpoint Management，兩者均支援 vPro。",
    "保固服務": "HP 提供 3 年全球保固，ThinkPad 提供 3 年 Premier Support，均可加購延伸保固。",
    "價格定位": "HP EliteBook 855 約 NT$45,000 起；ThinkPad X1 Carbon 約 NT$55,000 起，定位略高。",
}

research_question = "HP EliteBook 和 Lenovo ThinkPad 在企業市場的主要差異"
print(f"研究題目：{research_question}")

## Step 1：Planner — 把大問題拆成子問題

Planner 的工作：將研究題目拆成 3 個具體可搜尋的子問題。  
**子問題的品質決定了整份報告的深度。**

In [ ]:
# =============================================
# TODO：完成 Planner（約 6 行）
#
# 提示：
#   1. 用 llm.invoke 請 LLM 拆解研究題目
#   2. 要求回傳 3 個子問題，每行一個
#   3. 用 splitlines() 解析，過濾空行，回傳 list[str]
# =============================================
def planner(question: str) -> list[str]:
    """將研究題目拆解為 3 個子問題"""
    pass  # 請實作這裡


sub_questions = planner(research_question)
print("拆解出的子問題：")
for i, q in enumerate(sub_questions or [], 1):
    print(f"  {i}. {q}")

## Step 2：Reader + Synthesizer — 摘要與合併

In [ ]:
def reader(sub_question: str) -> str:
    """搜尋子問題並摘要（這裡用模擬資料）"""
    # 找最相關的 mock 資料（簡單關鍵字比對）
    for key, data in MOCK_DATA.items():
        if any(k in sub_question for k in ["規格", "硬體", "CPU", "RAM"]):
            return MOCK_DATA["硬體規格"]
        if any(k in sub_question for k in ["安全", "Security"]):
            return MOCK_DATA["安全性"]
        if any(k in sub_question for k in ["管理", "IT"]):
            return MOCK_DATA["企業管理"]
        if any(k in sub_question for k in ["保固", "服務"]):
            return MOCK_DATA["保固服務"]
        if any(k in sub_question for k in ["價格", "成本", "費用"]):
            return MOCK_DATA["價格定位"]
    # 找不到關鍵字就回傳第一筆
    return list(MOCK_DATA.values())[0]

def synthesizer(question: str, sub_answers: list[dict]) -> str:
    """將子問題答案合併成完整報告"""
    content = "\n\n".join(
        f"【{a['question']}】\n{a['answer']}" for a in sub_answers
    )
    return llm.invoke([
        SystemMessage(content="你是研究分析師，請根據資料撰寫一份繁體中文摘要報告。"),
        HumanMessage(content=f"研究題目：{question}\n\n資料：\n{content}")
    ]).content

print("✓ Reader / Synthesizer 準備好了")

## Step 3：執行完整 Deep Research

In [ ]:
# 執行完整流程
print("=" * 50)
print(f"研究題目：{research_question}")
print("=" * 50)

if sub_questions:
    sub_answers = []
    for q in sub_questions:
        answer = reader(q)
        sub_answers.append({"question": q, "answer": answer})
        print(f"✓ 完成子問題：{q[:40]}...")

    report = synthesizer(research_question, sub_answers)
    print("\n" + "=" * 50)
    print("最終報告：")
    print("=" * 50)
    print(report)
else:
    print("請先完成 Planner！")